# 【Playground S6E8 - Predicting Smartphone Addiction】NN Residual Network 解説付き写経

- **コンペ**: [Predicting Smartphone Addiction（Playground Series S6E8）](https://www.kaggle.com/competitions/playground-series-s6e8)（2,701チーム・残り8日）
- **原著者**: Anthony Therrien（[@anthonytherrien](https://www.kaggle.com/anthonytherrien)）
- **元notebook**: <https://www.kaggle.com/code/anthonytherrien/predicting-smartphone-addict-nn-residual-network>
- **スコア**: Public Score **0.97123**（本日時点の公開最高スコア帯）・36 upvotes
- **実行時間**: 短時間（GPU）

> ⚠️ これは**学習目的の解説付き写し**です。原著のコードは変更していませんが、
> 元は**巨大な1セル**だったため、読みやすさのために**論理的なまとまりごとにセルを分割**し、
> 各コードセルの直前に日本語の解説Markdownセルを挿入しています。出力（実行結果）は含みません。

## 手法の概要

表形式（tabular）の2値分類を、**PyTorchの残差MLP（Residual MLP）** で解くnotebook。特徴は次の3点。

1. **前処理をリークなしで組む**: 学習/検証に分割した**あとで** `ColumnTransformer` を学習側のみに fit する。
2. **LightGBMの予測値を1本の特徴量として足す（スタッキング）**: 5-fold の out-of-fold 予測を作り、ニューラルネットの入力に1列追加する。
3. **残差ブロック付きMLP**で学習し、検証AUCが最良のエポックの重みを復元する（early stoppingの手動実装）。

そして最後に、外部の提出ファイル2本と自分のNN予測を**加重ブレンド**して `submission.csv` を書き出します。

## 評価指標

**タスク**: 各ユーザーが「スマホ依存（`addicted_label`）」かどうかを予測する**2値分類**。予測は0/1のラベルではなく**確率（信頼度スコア）**で提出します。

**指標**: **ROC AUC**（Receiver Operating Characteristic 曲線の下の面積）。コード中でも `roc_auc_score` で検証しています。

- **意味**: 「正例をランダムに1つ、負例をランダムに1つ選んだとき、モデルが正例のほうに高いスコアを付ける確率」。0.5がランダム、1.0が完璧。
- **計算方法**: 予測スコアで全サンプルを降順に並べ、閾値を動かしながら（真陽性率, 偽陽性率）をプロットした曲線の下面積。

**なぜこの指標か**: AUCは**予測値の順位だけ**を見る指標で、絶対値のキャリブレーション（0.7が本当に70%かどうか）を問いません。Playgroundの合成データは正例/負例の比率が偏りがちで、accuracyだと「全部多数派と答える」だけで高得点が出てしまいます。AUCならその手は通じません。また、閾値を選ばずにモデルの識別能力そのものを測れるため、閾値調整の巧拙がスコアに混ざりません。

**このnotebookの設計が指標をどう最適化しているか**:

- **損失は `BCEWithLogitsLoss`、選択基準は検証AUC**。損失そのものはAUCではありませんが（AUCは微分不可能なので直接最適化できない）、**エポックごとに検証AUCを測り、最良の重みを `deepcopy` で保存**することで、実質的にAUCで最良の状態を選んでいます。
- **順位さえ合えばよい**という指標の性質を利用して、最後のブレンドは確率の加重平均で済ませています（AUCはスケール不変なので、単調変換なら順位は壊れません）。
- **LGBMのOOF予測を特徴量に足す**ことで、木モデルが得意な「非線形な閾値の切り方」をNNに与え、単体のNNより順位付けを鋭くしています。

## 一点だけ、正直な注意

最終ブレンドの重みは `sub1=2.9`, `sub2=0.1`, **`nn=1e-4`** です。つまり**この notebook が丁寧に学習した NN の寄与は実質ゼロ**で、LBスコア 0.97123 のほぼ全てが外部提出ファイル `sub1` に由来します。
学ぶべきパイプラインとしては非常に質が高い一方、**スコアの出どころとしては「他人の提出の載せ替え」に近い**という点は、鵜呑みにせず区別して読むべきところです。


## セル1: ライブラリの読み込み

**何をしているか**: PyTorch（NN本体）、LightGBM（スタッキング用の木モデル）、scikit-learn（前処理・分割・評価）、numpy/pandas を読み込みます。

**なぜそうするのか**: このnotebookは「木モデル」と「ニューラルネット」を組み合わせるハイブリッド構成なので、両方のライブラリが必要です。

**用語**:
- `torch.nn` … ニューラルネットの層（Linear, Dropout, BatchNorm など）が入ったモジュール。
- `ColumnTransformer` … 「数値列にはこの処理、カテゴリ列にはあの処理」と**列ごとに違う前処理**をまとめて適用する道具。
- `StratifiedKFold` … 正例/負例の比率を各foldで保ったまま分割する交差検証。不均衡データでは必須です。


In [ ]:
# Import dependencies
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset


## セル2: 残差ブロック（ResidualBlock）の定義

**何をしているか**: 「線形層 → ReLU → Dropout → 線形層」を通した結果に、**入力そのもの（residual）を足し戻す**ブロックを定義しています。足したあとに BatchNorm と ReLU を掛けます。

**なぜそうするのか**: `features = features + residual` の一行が肝です。これは ResNet で有名な**スキップ接続**で、次の2つの効果があります。

1. **勾配が消えにくい**: 逆伝播のとき、勾配が変換の経路と「そのまま足した経路」の両方を通れるので、層を深くしても学習が進みます。
2. **恒等写像を学ぶのが簡単**: このブロックが「何もしない」のが最適なら、内部の重みをゼロにすれば済みます。深くしても性能が悪化しにくいのはこのためです。

**用語**:
- **Dropout** … 学習中にランダムに一部のユニットを無効化して、特定の経路に依存しすぎるのを防ぐ（過学習対策）。
- **BatchNorm（バッチ正規化）** … ミニバッチ内で出力の平均・分散を揃える。学習が安定し、学習率を大きめに取れます。表形式データでは特に効きます。


In [ ]:
# Define the residual block
class ResidualBlock(nn.Module):
    # Define initialization
    def __init__(self, hidden_dim, dropout):
        # Call parent constructor
        super().__init__()

        # Define layers
        self.linear1 = nn.Linear(hidden_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.BatchNorm1d(hidden_dim)

    # Define forward pass
    def forward(self, features):
        # Save residual
        residual = features

        # Transform features
        features = self.linear1(features)
        features = self.activation(features)
        features = self.dropout(features)
        features = self.linear2(features)

        # Add residual and normalize
        features = features + residual
        features = self.batch_norm(features)
        features = self.activation(features)

        # Return features
        return features


## セル3: 分類器本体（ResidualClassifier）の定義

**何をしているか**: 入力層（特徴量数 → 128次元）→ ReLU → Dropout → **残差ブロック2つ** → 出力層（128 → 1）という構成のネットワークです。出力は `squeeze(1)` で1次元にします。

**なぜそうするのか**: 表形式データでは、画像のような超深層は不要です。**残差ブロック2つ程度の中規模MLP**が、勾配ブースティングと戦える現実的なラインとされています。深くしすぎると、行数の少ない表データではすぐ過学習します。

**重要な注意**: 出力層は1つの数値（ロジット）を出すだけで、**シグモイドを掛けていません**。これは後で `BCEWithLogitsLoss` を使うためです（この関数がシグモイドを内部で持っており、数値的に安定した計算をしてくれます）。ここで自分でシグモイドを掛けてしまうと二重適用になるので、初学者がよく踏むバグです。


In [ ]:
# Define the residual classifier
class ResidualClassifier(nn.Module):
    # Define initialization
    def __init__(self, input_dim, hidden_dim, dropout):
        # Call parent constructor
        super().__init__()

        # Define model layers
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
        )
        self.output_layer = nn.Linear(hidden_dim, 1)

    # Define forward pass
    def forward(self, features):
        # Transform features
        features = self.input_layer(features)
        features = self.activation(features)
        features = self.dropout(features)
        features = self.blocks(features)

        # Return one logit per row
        return self.output_layer(features).squeeze(1)


## セル4: 乱数シードの固定と、前処理パイプラインの構築

**何をしているか**: 2つの関数を定義します。

- `set_seed(seed)`: Python標準の `random`、numpy、PyTorch（CPU/CUDA）の乱数を全部同じ種で初期化する。
- `create_preprocessor(features)`: 数値列とカテゴリ列を自動判別し、**数値列には「中央値で欠損補完 → 標準化」**、**カテゴリ列には「最頻値で補完 → one-hotエンコーディング」**を適用する `ColumnTransformer` を返す。

**なぜそうするのか**:

- **シード固定**: 再現性のためです。乱数が変わるとスコアも変わるので、「改善したのか、ただ運が良かったのか」が区別できなくなります。
- **中央値で補完する理由**: 平均は外れ値に引っ張られます。「1日のスマホ利用時間」のような右に裾の長い分布では、中央値のほうが代表値として堅牢です。
- **`add_indicator=True`**: 「この値は欠損していた」というフラグ列を追加します。**欠損していること自体が情報**である場合（例: 回答を拒否した人には傾向がある）に効きます。
- **標準化（StandardScaler）が必須な理由**: ニューラルネットは入力のスケールに敏感です。片方が0〜1、もう片方が0〜10000だと、勾配のバランスが崩れて学習が進みません。木モデルでは不要ですが、NNでは必須です。
- **`handle_unknown="ignore"`**: テストにだけ現れるカテゴリ値が来ても、エラーで落ちずに全ゼロベクトルとして扱います。


In [ ]:
# Set random seeds
def set_seed(seed):
    # Seed all random generators
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Seed CUDA when available
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Create the preprocessing pipeline
def create_preprocessor(features):
    # Detect categorical columns
    categorical_columns = features.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    # Detect numerical columns
    numerical_columns = features.select_dtypes(
        exclude=["object", "category"]
    ).columns.tolist()

    # Define numerical preprocessing
    numerical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", StandardScaler()),
        ]
    )

    # Define categorical preprocessing
    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    # Return combined preprocessing
    return ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, numerical_columns),
            ("cat", categorical_transformer, categorical_columns),
        ]
    )


## セル5: データの準備 — 「分割してから前処理する」

**何をしているか**: 学習データから `id` と目的変数を落として特徴量を作り、**先に train/valid に分割してから**、`preprocessor` を**学習側のみに `fit`** し、検証・テストには `transform` だけを適用します。

**なぜそうするのか**: ここが本notebookで最も重要な作法です。コメントにも `# Split before fitting preprocessing to prevent leakage` と書かれています。

もし全データで `fit` してから分割すると、標準化に使う平均・分散や、欠損補完に使う中央値に**検証データの情報が混ざります**。これを**データリーク（leakage）**と呼び、検証スコアだけが不当に良くなり、LBでは再現しません。「CVは高いのにLBが低い」典型的な原因の一つです。

**用語**:
- `fit` = データから統計量（平均・分散・中央値・カテゴリ一覧）を学ぶ / `transform` = 学んだ統計量で変換する。**testに `fit` してはいけない**、と覚えてください。
- `stratify=target` … 分割時に正例/負例の比率を保つ指定。不均衡データで検証スコアを安定させます。
- `astype(np.float32)` … PyTorchのデフォルトは float32 です。float64のまま渡すと型エラーになります。


In [ ]:
# Prepare train, validation, and test matrices
def preprocess_data(train_df, test_df, target_column, seed):
    # Separate predictors and target
    features = train_df.drop(columns=["id", target_column])
    target = train_df[target_column].to_numpy(dtype=np.float32)
    test_features = test_df.drop(columns=["id"])

    # Split before fitting preprocessing to prevent leakage
    train_features, valid_features, train_target, valid_target = train_test_split(
        features,
        target,
        test_size=0.1,
        random_state=seed,
        stratify=target,
    )

    # Fit preprocessing on the training fold only
    preprocessor = create_preprocessor(train_features)
    train_matrix = preprocessor.fit_transform(train_features)
    valid_matrix = preprocessor.transform(valid_features)
    test_matrix = preprocessor.transform(test_features)

    # Convert matrices to float32
    train_matrix = np.asarray(train_matrix, dtype=np.float32)
    valid_matrix = np.asarray(valid_matrix, dtype=np.float32)
    test_matrix = np.asarray(test_matrix, dtype=np.float32)

    # Return prepared data
    return (
        train_matrix,
        valid_matrix,
        train_target,
        valid_target,
        test_matrix,
        test_df["id"].copy(),
    )


## セル6: LightGBMのOOF予測を特徴量として足す（スタッキング）

**何をしているか**: 学習データを StratifiedKFold で5分割し、各foldについて「そのfold以外で学習した LightGBM」で当該foldを予測します（**out-of-fold予測**）。検証・テストに対しては5モデルの平均を使います。最後に、その確率を**1列の新しい特徴量**として全行列に `column_stack` で追加します。

**なぜそうするのか**:

- **なぜ木モデルの出力をNNに足すのか**: 勾配ブースティングは「特徴量Aが3.5を超えたら…」といった**軸に平行な閾値の切り方**が得意で、MLPはそれを苦手とします。逆にMLPは特徴量間のなめらかな相互作用が得意です。片方の出力をもう片方の入力に渡すことで、**互いの弱点を補う**わけです。これがスタッキング（stacking）の基本発想です。
- **なぜ out-of-fold なのか**: もし全学習データで学習したLGBMで学習データを予測して特徴量にすると、その列には**正解が漏れています**（LGBMはその行を覚えている）。NNは「その列だけ見ればいい」と学習してしまい、テストで崩壊します。「予測する行は、その行を学習に使っていないモデルで予測する」——これがOOFの鉄則です。
- **なぜ検証・テストは平均なのか**: 検証・テスト行はどのfoldでも未使用なので、5モデル全部で予測して平均すれば分散が減り、より安定します（`/ folds` で割っている部分）。

**読みどころ**: fold ごとのAUC、OOF全体のAUC、検証AUCを印字しています。**OOF AUC と 検証AUC が大きく離れていたら、どこかがおかしい**というチェックになります。


In [ ]:
# Add leakage-safe LightGBM probability features
def add_lightgbm_features(
    train_matrix,
    valid_matrix,
    test_matrix,
    train_target,
    valid_target,
    seed,
    folds=5,
):
    # Initialize stacked prediction features
    train_lgbm_feature = np.zeros(len(train_matrix), dtype=np.float32)
    valid_lgbm_feature = np.zeros(len(valid_matrix), dtype=np.float64)
    test_lgbm_feature = np.zeros(len(test_matrix), dtype=np.float64)

    # Create stratified folds for out-of-fold predictions
    splitter = StratifiedKFold(
        n_splits=folds,
        shuffle=True,
        random_state=seed,
    )

    # Train one LightGBM model per fold
    for fold, (fit_indices, oof_indices) in enumerate(
        splitter.split(train_matrix, train_target),
        start=1,
    ):
        # Define the fold model
        model = LGBMClassifier(
            objective="binary",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=seed + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        # Fit without using the held-out fold
        model.fit(
            train_matrix[fit_indices],
            train_target[fit_indices],
            eval_set=[
                (train_matrix[oof_indices], train_target[oof_indices])
            ],
            eval_metric="auc",
            callbacks=[],
        )

        # Create out-of-fold and inference features
        train_lgbm_feature[oof_indices] = model.predict_proba(
            train_matrix[oof_indices]
        )[:, 1]
        valid_lgbm_feature += model.predict_proba(valid_matrix)[:, 1] / folds
        test_lgbm_feature += model.predict_proba(test_matrix)[:, 1] / folds

        # Print fold performance
        fold_auc = roc_auc_score(
            train_target[oof_indices],
            train_lgbm_feature[oof_indices],
        )
        print(f"LightGBM fold {fold:02d} AUC: {fold_auc:.6f}")

    # Print aggregate feature performance
    oof_auc = roc_auc_score(train_target, train_lgbm_feature)
    valid_auc = roc_auc_score(valid_target, valid_lgbm_feature)
    print(f"LightGBM OOF AUC: {oof_auc:.6f}")
    print(f"LightGBM validation AUC: {valid_auc:.6f}")

    # Append one LightGBM probability column to every matrix
    train_matrix = np.column_stack(
        [train_matrix, train_lgbm_feature]
    ).astype(np.float32)
    valid_matrix = np.column_stack(
        [valid_matrix, valid_lgbm_feature]
    ).astype(np.float32)
    test_matrix = np.column_stack(
        [test_matrix, test_lgbm_feature]
    ).astype(np.float32)

    # Return augmented feature matrices
    return train_matrix, valid_matrix, test_matrix


## セル7: DataLoader の作成と、確率予測の関数

**何をしているか**: numpy配列をPyTorchのテンソルに変換し、`DataLoader` でミニバッチに小分けします。ラベルの有無で「学習用ローダー」と「推論用ローダー」を作り分けています。加えて、モデルを評価モードにして確率を吐く `predict_probabilities` を定義します。

**なぜそうするのか**:

- **`model.eval()` の意味**: Dropout を無効化し、BatchNorm を「学習中に蓄積した移動平均」を使うモードに切り替えます。**これを忘れると推論のたびに結果が変わり、性能も落ちます**。初学者が最も踏みやすい罠の一つです。
- **`torch.no_grad()` の意味**: 勾配計算用の情報を保持しなくなるので、メモリを大幅に節約し高速化します。推論では勾配が不要なので必ず付けます。
- **`torch.sigmoid(logits)`**: モデルの生出力（ロジット）を0〜1の確率に変換します。学習時は `BCEWithLogitsLoss` が内部でやってくれるので、**明示的に掛けるのは推論のときだけ**です。
- **`batch_size=4096` / `8192` と大きい理由**: 表形式データは1行が軽いため、大きなバッチのほうがGPUを効率よく使えます。画像の常識（32〜64）とは違います。
- **`pin_memory`**: CPU→GPUの転送を速くするオプション。GPUがある時だけTrueにしています。


In [ ]:
# Create a tensor data loader
def create_loader(features, target=None, batch_size=4096, shuffle=False):
    # Convert features to a tensor
    feature_tensor = torch.from_numpy(features)

    # Create an inference loader
    if target is None:
        return DataLoader(
            feature_tensor,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=torch.cuda.is_available(),
        )

    # Convert target to a tensor
    target_tensor = torch.from_numpy(target)

    # Create a supervised loader
    dataset = TensorDataset(feature_tensor, target_tensor)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=torch.cuda.is_available(),
    )


# Predict positive-class probabilities
def predict_probabilities(model, loader, device):
    # Set evaluation mode
    model.eval()

    # Initialize predictions
    predictions = []

    # Disable gradient tracking
    with torch.no_grad():
        # Iterate batches
        for batch in loader:
            # Support supervised and inference loaders
            batch_features = batch[0] if isinstance(batch, list) else batch
            batch_features = batch_features.to(device, non_blocking=True)

            # Convert logits to probabilities
            logits = model(batch_features)
            probabilities = torch.sigmoid(logits)
            predictions.append(probabilities.cpu().numpy())

    # Concatenate predictions
    return np.concatenate(predictions)


## セル8: 学習ループ — 検証AUCが最良の重みを保存する

**何をしているか**: `BCEWithLogitsLoss` と `AdamW` で24エポック学習し、**各エポックの終わりに検証AUCを計算**、これまでの最良を更新したら `copy.deepcopy(model.state_dict())` で重みのコピーを保存します。最後に最良の重みを復元して返します。

**なぜそうするのか**:

- **early stopping の手動実装**: 「学習は最後まで回すが、採用するのは検証が最良だったエポックの重み」という戦略です。学習が進みすぎて過学習に入っても、その前の一番良い状態に巻き戻せます。
- **なぜ `deepcopy` が必要か**: `state_dict()` が返すテンソルは**モデルの重みと同じメモリを参照している**ため、そのまま持っておくと次のエポックで上書きされ、「保存したはずの最良重み」が最新の重みに化けます。この一行がないと静かにバグります。
- **`AdamW` と `weight_decay=1e-3`**: AdamW は重み減衰（L2正則化）を勾配とは別枠で正しく適用する版のAdam。素のAdamで `weight_decay` を指定するより正則化が効きます。
- **`optimizer.zero_grad(set_to_none=True)`**: PyTorchは勾配を累積するので、毎ステップ必ずリセットが必要です。`set_to_none=True` はゼロ埋めではなくNoneにするぶん、わずかに速くメモリ効率も良くなります。
- **`BCEWithLogitsLoss`**: シグモイド + 交差エントロピーを一体で計算し、数値的に安定します（log(0)によるNaNを避けられる）。

**用語**: `total_loss += loss.item() * batch_features.size(0)` は、バッチサイズが不揃いでも正しい平均が出るように**行数で重み付けして合計**しています。細かいですが正しい書き方です。


In [ ]:
# Train the classifier
def train_model(
    model,
    train_loader,
    valid_loader,
    valid_target,
    device,
    epochs,
    learning_rate,
):
    # Define binary classification loss
    criterion = nn.BCEWithLogitsLoss()

    # Define optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-3,
    )

    # Initialize best validation state
    best_auc = -np.inf
    best_state = None

    # Run training epochs
    for epoch in range(epochs):
        # Set training mode
        model.train()

        # Initialize loss totals
        total_loss = 0.0
        total_rows = 0

        # Iterate training batches
        for batch_features, batch_target in train_loader:
            # Move batch to device
            batch_features = batch_features.to(device, non_blocking=True)
            batch_target = batch_target.to(device, non_blocking=True)

            # Update model weights
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            loss = criterion(logits, batch_target)
            loss.backward()
            optimizer.step()

            # Accumulate row-weighted loss
            total_loss += loss.item() * batch_features.size(0)
            total_rows += batch_features.size(0)

        # Evaluate validation ROC AUC
        valid_predictions = predict_probabilities(
            model,
            valid_loader,
            device,
        )
        valid_auc = roc_auc_score(valid_target, valid_predictions)

        # Print epoch metrics
        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train BCE: {total_loss / total_rows:.6f} | "
            f"Valid AUC: {valid_auc:.6f}"
        )

        # Save an independent copy of the best weights
        if valid_auc > best_auc:
            best_auc = valid_auc
            best_state = copy.deepcopy(model.state_dict())

    # Restore best weights
    model.load_state_dict(best_state)

    # Print best score
    print(f"Best validation AUC: {best_auc:.6f}")

    # Return trained model
    return model


## セル9: 複数予測の加重ブレンドと提出ファイルの書き出し

**何をしているか**: 名前つきの辞書（`prediction_configs`）で複数の予測と重みを受け取り、**行数が一致するかを検証したうえで**加重平均し、`np.clip(..., 0.0, 1.0)` で範囲を整えて `submission.csv` に保存します。

**なぜそうするのか**:

- **なぜブレンドが効くのか**: 異なるモデルは異なる誤りをします。誤りが独立に近いほど、平均すると誤差が打ち消し合い、単体のどれよりも良くなることがあります（アンサンブルの基本原理）。
- **行数チェックの価値**: `if len(predictions) != len(test_ids): raise ValueError(...)` の3行が、「idの順序がずれたまま平均してしまう」という**気づきにくく致命的なバグ**を防ぎます。ブレンド系notebookで最も多い事故がこれです。
- **`total_weight` で割る理由**: 重みの合計が1でなくても正しい加重平均になるようにするためです。重みを試行錯誤する際に、いちいち正規化しなくて済みます。

**用語**: `np.clip` は値を指定範囲に収める関数。確率として出す以上、数値誤差で1.0000001 のような値が出ても弾いておくのが安全です。


In [ ]:
# Blend predictions and save the Kaggle submission
def blend_predictions(
    test_ids,
    prediction_configs,
    target_column,
    output_path,
):
    # Initialize blend totals
    weighted_predictions = np.zeros(len(test_ids), dtype=np.float64)
    total_weight = 0.0

    # Blend every prediction source
    for prediction_name, config in prediction_configs.items():
        # Extract predictions and weight
        predictions = np.asarray(config["predictions"], dtype=np.float64)
        weight = float(config["weight"])

        # Validate prediction length
        if len(predictions) != len(test_ids):
            raise ValueError(
                f"{prediction_name} has {len(predictions)} rows; "
                f"expected {len(test_ids)}."
            )

        # Add weighted predictions
        weighted_predictions += predictions * weight
        total_weight += weight

        # Print blend information
        print(f"Blending {prediction_name} | Weight: {weight}")

    # Validate total weight
    if total_weight <= 0.0:
        raise ValueError("The total blend weight must be positive.")

    # Compute final probabilities
    final_predictions = weighted_predictions / total_weight

    # Create submission frame
    submission = pd.DataFrame(
        {
            "id": test_ids.to_numpy(),
            target_column: np.clip(final_predictions, 0.0, 1.0),
        }
    )

    # Save submission
    submission.to_csv(output_path, index=False)

    # Print confirmation
    print(f"Submission saved to {output_path}")


## セル10: main関数 — パイプライン全体の組み立てと最終ブレンド

**何をしているか**: これまで定義した部品を順に呼び出します。データ読み込み → スキーマ検証 → 前処理 → LGBM特徴の追加 → ローダー作成 → モデル構築（hidden=128, dropout=0.2）→ 24エポック学習 → テスト予測 → 外部提出2本と自分のNNをブレンド → 保存。

**なぜそうするのか（設計面の良い点）**:

- **スキーマ検証**: `set(test_df.columns) != expected_test_columns` で、train/test の列の食い違いを最初に検出します。列がずれたまま処理を進めると、原因不明のスコア低下として現れます。
- **idの一致検証**: 外部の提出ファイルについても `np.array_equal(submission["id"], test_ids)` で順序が同じことを確認しています。前セルの行数チェックより厳しく、**中身の並び順まで**見ています。
- **`main()` に処理を閉じ込める**: グローバル変数が散らからず、上から順に読めば流れが分かります。関数に切り分ける設計は、ここまでの各セルを部品として再利用できる形にしてくれています。

**⚠️ ここは批判的に読むべき点**:

```python
"sub1": {"weight": 2.9},
"sub2": {"weight": 0.1},
"nn":   {"weight": 1e-4},   # ← 実質ゼロ
```

自分で学習したNNの重みは **0.0001**、つまり最終予測への寄与は事実上ありません。**このnotebookのLBスコア 0.97123 は、外部提出ファイル `sub1` のスコアとほぼ同一**と考えるべきです。

タイトルは「NN Residual Network」ですが、スコアを作っているのはNNではありません。**タイトル・スコア・実際の寄与が食い違っている**典型例で、Kaggleの公開notebookを読むときに常に確認すべきポイントです。
とはいえ、前処理・OOFスタッキング・学習ループの実装は非常に丁寧で、**パイプラインの教材としての価値は高い**ままです。学ぶ対象と、信じるスコアは分けて考えましょう。


In [ ]:
# Define the main function
def main():
    # Define configuration
    target_column = "addicted_label"
    competition_path = Path(
        "/kaggle/input/competitions/playground-series-s6e8"
    )
    blend_path = Path(
        "/kaggle/input/datasets/anthonytherrien/predicting-smartphone-addiction-vault"
    )
    output_path = Path("submission.csv")
    seed = 42

    # Set random seeds
    set_seed(seed)

    # Load competition data
    train_df = pd.read_csv(competition_path / "train.csv")
    test_df = pd.read_csv(competition_path / "test.csv")

    # Validate expected schemas
    expected_test_columns = set(train_df.columns) - {target_column}
    if set(test_df.columns) != expected_test_columns:
        raise ValueError("Train and test feature columns do not match.")

    # Prepare data
    (
        train_matrix,
        valid_matrix,
        train_target,
        valid_target,
        test_matrix,
        test_ids,
    ) = preprocess_data(
        train_df,
        test_df,
        target_column,
        seed,
    )

    # Add LightGBM predictions as a stacked feature
    train_matrix, valid_matrix, test_matrix = add_lightgbm_features(
        train_matrix=train_matrix,
        valid_matrix=valid_matrix,
        test_matrix=test_matrix,
        train_target=train_target,
        valid_target=valid_target,
        seed=seed,
    )

    # Create loaders
    train_loader = create_loader(
        train_matrix,
        train_target,
        batch_size=4096,
        shuffle=True,
    )
    valid_loader = create_loader(
        valid_matrix,
        batch_size=8192,
    )
    test_loader = create_loader(
        test_matrix,
        batch_size=8192,
    )

    # Select compute device
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print(f"Using device: {device}")
    print(f"Processed feature count: {train_matrix.shape[1]}")

    # Create classifier
    model = ResidualClassifier(
        input_dim=train_matrix.shape[1],
        hidden_dim=128,
        dropout=0.2,
    ).to(device)

    # Train classifier
    model = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        valid_target=valid_target,
        device=device,
        epochs=24,
        learning_rate=1e-3,
    )

    # Predict test probabilities
    test_predictions = predict_probabilities(
        model,
        test_loader,
        device,
    )

    # Load external submission files
    submission_one = pd.read_csv(blend_path / "submission.csv")
    submission_two = pd.read_csv(blend_path / "submission (1).csv")

    # Validate external submission ids
    for submission_name, submission in {
        "sub1": submission_one,
        "sub2": submission_two,
    }.items():
        if not np.array_equal(submission["id"].to_numpy(), test_ids.to_numpy()):
            raise ValueError(f"Ids in {submission_name} do not match test.csv.")

    # Define prediction blend
    prediction_configs = {
        "sub1": {
            "predictions": submission_one[target_column].to_numpy(),
            "weight": 2.9,
        },
        "sub2": {
            "predictions": submission_two[target_column].to_numpy(),
            "weight": 0.1,
        },
        "nn": {
            "predictions": test_predictions,
            "weight": 1e-4,
        },
    }

    # Blend predictions and save submission
    blend_predictions(
        test_ids=test_ids,
        prediction_configs=prediction_configs,
        target_column=target_column,
        output_path=output_path,
    )


# Call the main function
if __name__ == "__main__":
    main()
